# 🍷 Global Alcohol Consumption Explorer

**IE University — Group Capstone Project**

**Team:** Nityaan · Oscar · Nico

This notebook contains the **whole project in one file**: it loads the data,
cleans it, analyses it, and draws the same interactive charts as our dashboard,
with explanations along the way. All values are measured in **litres of pure
alcohol per person per year**.

### How to use this notebook
Run the cells from top to bottom (in Google Colab: **Runtime ▸ Run all**).
The notebook is split into the same three "sections" as the dashboard
(Overview, Country Explorer, Beverage Analysis). In a few places you can change
a simple variable (the year, the country, the drink) and re-run that cell to
explore different views.

## 1. The problem and our questions

Alcohol consumption varies a lot around the world and over time. Understanding
these patterns is useful for public-health and social research. We use open
data to answer three questions:

1. **How does alcohol consumption differ across countries?**
2. **How has alcohol consumption changed over time?**
3. **Which beverage type — beer, wine or spirits — dominates in each country?**

## 2. The data and its source

We use four open datasets from **Our World in Data** (based on WHO figures).
Each file has the same columns: `Entity`, `Code`, `Year`, and one value column.

| Dataset | Indicator | Years |
|---|---|---|
| total | total alcohol | 2000–2020 |
| beer | beer | 1960–2022 |
| wine | wine | 1960–2022 |
| spirits | spirits | 1960–2022 |

Source: Our World in Data — Alcohol Consumption
(https://ourworldindata.org/alcohol-consumption).

> **Important:** the *total* indicator and the *beer/wine/spirits* indicators
> come from different data series with different year ranges and coverage, so
> **beer + wine + spirits does not equal the total**. We treat them as separate
> measures and never add them together.

## 3. Setup — import the libraries

We only need two main libraries: **pandas** for the data and **plotly** for the
interactive charts. Both are already installed in Google Colab.

In [ ]:
import os
import pandas as pd
import plotly.express as px

print("Libraries loaded.")

## 4. Load the data

The notebook first looks for a local `data/` folder (this is the case when you
run it in PyCharm or Jupyter next to the CSV files). If that folder is not found
— for example on Google Colab — it downloads the files directly from our public
GitHub repository instead. Either way the data loads automatically.

While loading, we also rename each long value column to a short name
(`total`, `beer`, `wine`, `spirits`) so the rest of the code is easy to read.

In [ ]:
# Where to download the files from if there is no local "data" folder.
GITHUB_BASE = "https://raw.githubusercontent.com/nityaan1/global-alcohol-explorer/main/data/"

# File name for each indicator (the key is the short name we will use).
FILES = {
    "total":   "total-alcohol-consumption-per-capita-litres-of-pure-alcohol.csv",
    "beer":    "beer-consumption-per-person.csv",
    "wine":    "wine-consumption-per-capita.csv",
    "spirits": "spirits-consumption-per-person.csv",
}

def load_csv(key):
    # Load one dataset and rename its value column to the short name.
    local_path = os.path.join("data", FILES[key])
    source = local_path if os.path.exists(local_path) else GITHUB_BASE + FILES[key]
    df = pd.read_csv(source)
    value_column = df.columns[3]           # the 4th column holds the numbers
    df = df.rename(columns={value_column: key})
    return df

# Load the total dataset and the three beverage datasets.
total   = load_csv("total")
beer    = load_csv("beer")
wine    = load_csv("wine")
spirits = load_csv("spirits")

print("total dataset shape:", total.shape)
total.head()

## 5. Data processing and cleaning

Three simple steps:

1. **Merge** the three beverage files into one table (they share the same
   countries and years).
2. **Data-quality check** — look for missing values and duplicate rows.
3. **Separate real countries from regional aggregates.** Real countries have a
   3-letter ISO code (e.g. `FRA`), while aggregates such as *World* or World
   Bank regions use longer codes (e.g. `OWID_WRL`, `WB_EAP`). We keep aggregates
   only for the "world average" figure; rankings and maps use countries only.

In [ ]:
# 1. Merge the three beverage files on Entity + Code + Year.
beverages = (beer
             .merge(wine, on=["Entity", "Code", "Year"])
             .merge(spirits, on=["Entity", "Code", "Year"]))

# 2. Data-quality check: count missing values and duplicate rows in each file.
for name, df in [("total", total), ("beer", beer), ("wine", wine), ("spirits", spirits)]:
    print(f"{name:8} missing values: {df.isna().sum().sum():>3} | "
          f"duplicate rows: {df.duplicated().sum():>3}")

# 3. Helper functions to split countries from aggregates.
def only_countries(df):
    return df[df["Code"].str.len() == 3].copy()

def only_aggregates(df):
    return df[df["Code"].str.len() != 3].copy()

total_countries    = only_countries(total)
beverage_countries = only_countries(beverages)

print("\nCountries in total :", total_countries["Entity"].nunique())
print("Aggregates in total:", only_aggregates(total)["Entity"].nunique())
print("Merged beverages   :", beverages.shape)

## 6. Section 1 — Overview

**Question: how does total consumption differ across countries?**

Pick a year, then look at the world map, the top-10 countries, and the headline
numbers (world average, highest, lowest). Change `year` below and re-run to see
a different year (2000–2020).

In [ ]:
# >>> Change this year (any value from 2000 to 2020) and run the cell again. <<<
year = 2020

# Keep only the rows for the chosen year (countries only, no aggregates).
countries_year = total_countries[total_countries["Year"] == year]

# Headline numbers. The world average comes from the official "World" row.
world_row = only_aggregates(total)
world_row = world_row[(world_row["Entity"] == "World") & (world_row["Year"] == year)]
world_value = world_row["total"].iloc[0] if not world_row.empty else None

highest = countries_year.loc[countries_year["total"].idxmax()]
lowest  = countries_year.loc[countries_year["total"].idxmin()]

print(f"Year {year}")
if world_value is not None:
    print(f"  World average : {world_value:.1f} L")
print(f"  Highest       : {highest['Entity']} ({highest['total']:.1f} L)")
print(f"  Lowest        : {lowest['Entity']} ({lowest['total']:.1f} L)")

# World map: colour each country by its total consumption.
px.choropleth(
    countries_year, locations="Code", color="total", hover_name="Entity",
    color_continuous_scale="Reds", labels={"total": "Litres"},
    title=f"Total alcohol consumption by country, {year}",
).show()

In [ ]:
# Top 10 countries for the chosen year.
top10 = countries_year.sort_values("total", ascending=False).head(10)
px.bar(
    top10.sort_values("total"), x="total", y="Entity", orientation="h",
    labels={"total": "Litres of pure alcohol", "Entity": ""},
    title=f"Top 10 countries, {year}",
).show()

## 7. Section 2 — Country Explorer

**Question: how has consumption changed over time?**

Pick one country and look at its total-consumption trend (2000–2020), its
beer/wine/spirits history (1960–2022) and its most recent drink mix. Change
`country` below and re-run.

In [ ]:
# >>> Change this country and run the cell again. <<<
country = "France"

# Total consumption trend (2000-2020).
country_total = total_countries[total_countries["Entity"] == country]
px.line(
    country_total, x="Year", y="total", markers=True,
    labels={"total": "Litres of pure alcohol"},
    title=f"Total alcohol consumption in {country} (2000-2020)",
).show()

In [ ]:
# Beer, wine and spirits for the same country.
country_bev = beverage_countries[beverage_countries["Entity"] == country]

if country_bev.empty:
    # A few countries (e.g. Saudi Arabia) have no beverage breakdown.
    print(f"No beer/wine/spirits data is available for {country}.")
else:
    # Reshape to "long" form so Plotly can draw one line per drink.
    long_bev = country_bev.melt(
        id_vars="Year", value_vars=["beer", "wine", "spirits"],
        var_name="Drink", value_name="Litres",
    )
    px.line(long_bev, x="Year", y="Litres", color="Drink",
            title=f"Beer, wine and spirits in {country} (1960-2022)").show()

    # Drink mix in the most recent available year.
    latest_year = int(country_bev["Year"].max())
    latest = country_bev[country_bev["Year"] == latest_year]
    mix = pd.DataFrame({
        "Drink": ["beer", "wine", "spirits"],
        "Litres": [latest["beer"].iloc[0], latest["wine"].iloc[0],
                   latest["spirits"].iloc[0]],
    })
    px.bar(mix, x="Drink", y="Litres", color="Drink",
           labels={"Litres": "Litres of pure alcohol"},
           title=f"Drink mix in {country} ({latest_year})").show()

## 8. Section 3 — Beverage Analysis

**Question: which beverage type dominates in each country?**

Rank countries by a chosen drink, see which drink is the most consumed in the
largest number of countries, and look at the global trend of each drink over
time. Change `drink` below and re-run.

In [ ]:
# Most recent year available in the beverage data.
latest_year = int(beverage_countries["Year"].max())
latest = beverage_countries[beverage_countries["Year"] == latest_year]

# >>> Change the drink ("beer", "wine" or "spirits") and run the cell again. <<<
drink = "beer"

top10 = latest.sort_values(drink, ascending=False).head(10)
px.bar(top10.sort_values(drink), x=drink, y="Entity", orientation="h",
       labels={drink: "Litres of pure alcohol", "Entity": ""},
       title=f"Top 10 countries by {drink} ({latest_year})").show()

In [ ]:
# Which drink is the most consumed in each country (of the three tracked types)?
dominant = latest[["beer", "wine", "spirits"]].idxmax(axis=1)
counts = dominant.value_counts().reset_index()
counts.columns = ["Drink", "Number of countries"]
print(counts.to_string(index=False))

px.bar(counts, x="Drink", y="Number of countries", color="Drink",
       title=f"Which drink dominates? ({latest_year})").show()

In [ ]:
# Global average of each drink over time (averaged across reporting countries).
yearly_avg = (beverage_countries
              .groupby("Year")[["beer", "wine", "spirits"]]
              .mean()
              .reset_index())
long_avg = yearly_avg.melt(id_vars="Year", value_vars=["beer", "wine", "spirits"],
                           var_name="Drink", value_name="Litres")
px.line(long_avg, x="Year", y="Litres", color="Drink",
        title="Global average by drink over time (1960-2022)").show()

print("Note: fewer countries reported in the early years, so early values are "
      "less representative than recent ones.")

## 9. Key findings

A few headline results, taken directly from the data above:

- **Consumption varies widely across countries.** In 2020 the highest total
  consumption was **Romania (about 16.8 L)** and the lowest was **Kuwait
  (about 0.0 L)**, against a world average of roughly **4.9 L**.
- **Beer is the most common leading drink.** In 2022, beer was the most-consumed
  beverage type in **103** countries, spirits in **64**, and wine in **20**.
- **Different drinks lead in different places.** In 2022 the top country for each
  drink was **Czechia (beer)**, **Portugal (wine)** and **Saint Lucia (spirits)**.

## 10. Conclusion

The project set out to answer three questions, and this notebook answers all
three:

1. **Differences across countries** — consumption is very uneven (Romania near
   16.8 L versus Kuwait near 0 L in 2020), as the world map and rankings show.
2. **Change over time** — the trend charts show how each country's total and
   beverage consumption has moved since 1960 (beverages) and 2000 (total).
3. **Which drink dominates** — beer leads in the largest number of countries,
   followed by spirits and then wine.

The main thing we learned is that careful data processing plus a few
well-chosen charts can turn four raw CSV files into a clear, interactive story
about global drinking patterns.

## 11. Limitations and notes on method

- The **total** and **beer/wine/spirits** indicators are separate measures with
  different coverage and year ranges, so **they must not be added together**.
- Beverage figures are *recorded* consumption only (they exclude home-made or
  unrecorded alcohol).
- A value of `0` means "reported as zero" and cannot be told apart from
  "not reported".
- The global average-by-drink chart averages whichever countries reported in
  each year; fewer countries reported in the early years.

### Possible future improvements
- Let the user compare two or more countries on the same chart.
- Add more recent years as updated data becomes available.
- Bring in population or income data to explore possible relationships.